# NSL23 low-resource alphabet recognition

This notebook reproduces the **tables and landmark inference** claimed in:

> Low-Resource Nepali Sign Language Alphabet Recognition on NSL23: Comparing RGB Video Models and Hand Landmarks for Webcam Use

It does **not** download the ~10 GB NSL23 video archives and it does **not** retrain the 18 RGB runs (those used an Apple M4 and a frozen ImageNet MobileNetV2 encoder).

What you can run here:

1. Load the audited 30-run result tables (3 RGB models + 2 landmark models, 2 signer-held-out folds, 3 seeds).
2. Reload a committed landmark BiLSTM checkpoint and classify cached 16-frame MediaPipe vectors (no videos required).
3. Inspect the vowel vs consonant gap that drives the paper's interpretation.

The live webcam loop is `scripts/webcam_demo.py` on a local machine. Colab does not attach to your camera in a useful way for that script.

Dataset credit: Sunuwar, Borah, and Kharga, *Data in Brief* (2024), Zenodo [10.5281/zenodo.10478554](https://doi.org/10.5281/zenodo.10478554), CC BY 4.0. This notebook does not redistribute the videos.


## 1. Setup

Set `CLONE_REPO=True` on Colab if the repository is not already the working directory.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, platform

IN_COLAB = "google.colab" in sys.modules
CLONE_REPO = IN_COLAB
REPO_URL = "https://github.com/twasanon/nsl23-low-resource-alphabet.git"
ROOT = Path("/content/nsl23-low-resource-alphabet") if IN_COLAB else Path.cwd()

if CLONE_REPO and IN_COLAB:
    if not (ROOT / "src" / "nsl23" / "models.py").exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)])
    os.chdir(ROOT)
else:
    os.chdir(ROOT)

assert (ROOT / "src" / "nsl23" / "models.py").is_file(), "Run from the repo root or enable CLONE_REPO."
sys.path.insert(0, str(ROOT / "src"))
print({"in_colab": IN_COLAB, "root": str(Path.cwd()), "python": platform.python_version()})
try:
    from IPython.display import display
except ImportError:
    display = print


In [ ]:
import sys, subprocess

pkgs = [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "mediapipe==0.10.14",
]
if "google.colab" in sys.modules:
    pkgs.append("torch")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
print({"numpy": np.__version__, "pandas": pd.__version__, "torch": torch.__version__, "device": "cpu"})


## 2. Paper tables (30-run matrix)

Each seed pools S1 and S2 (196 unique held-out clips). Architecture rows are mean ± sample SD over seeds 17, 42, and 73. Chance accuracy for 49 classes is $1/49 \approx 2.0\%$.


In [ ]:
from pathlib import Path
import pandas as pd

results = Path("artifacts/results")
model = pd.read_csv(results / "model_summary.csv").set_index("model")
order = ["single_frame_cnn", "cnn_bilstm", "cnn_transformer", "landmark_mlp", "landmark_bilstm"]
labels = {
    "single_frame_cnn": "Single-frame CNN",
    "cnn_bilstm": "CNN–BiLSTM",
    "cnn_transformer": "CNN–Transformer",
    "landmark_mlp": "Landmark MLP",
    "landmark_bilstm": "Landmark BiLSTM",
}

def pm(mean, sd):
    return f"{100 * mean:.1f}% ± {100 * sd:.1f}%"

rows = []
for name in order:
    r = model.loc[name]
    rows.append({
        "Model": labels[name],
        "Accuracy": pm(r.accuracy_mean, r.accuracy_sd),
        "Macro-F1": pm(r.macro_f1_mean, r.macro_f1_sd),
    })
display(pd.DataFrame(rows))
print("Highest accuracy:", labels[model.accuracy_mean.idxmax()])
assert abs(model.loc["landmark_bilstm", "accuracy_mean"] - 0.24829931972789118) < 1e-6
assert abs(model.loc["single_frame_cnn", "accuracy_mean"] - 0.05782312925170068) < 1e-6
print("Table matches the frozen 30-run summary.")


In [ ]:
sign = pd.read_csv(results / "sign_type_summary.csv")
sign["sign_type"] = sign["sign_type"].str.lower()
table = []
for name in order:
    cons = sign[(sign.model == name) & (sign.sign_type == "consonant")].iloc[0]
    vow = sign[(sign.model == name) & (sign.sign_type == "vowel")].iloc[0]
    table.append({
        "Model": labels[name],
        "Consonant acc.": pm(cons.accuracy_mean, cons.accuracy_sd),
        "Vowel acc.": pm(vow.accuracy_mean, vow.accuracy_sd),
    })
display(pd.DataFrame(table))
print("The landmark gain is mostly vowels. Consonants stay near 10% even for the BiLSTM.")


## 3. Landmark inference without videos

Cached files in `data/landmark_cache_primary/` are 16 × 64 arrays: 21 MediaPipe xyz points, wrist-relative and scaled by the middle-MCP distance, plus a detection flag. The fold CSVs may contain absolute paths from the original Mac; the loader falls back to this directory by `sample_id`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from nsl23.models import build_model

ckpt = Path("artifacts/runs/primary/heldout_s1/landmark_bilstm/seed_42/best_checkpoint.pt")
payload = torch.load(ckpt, map_location="cpu", weights_only=False)
net = build_model(payload["model_name"], payload["experiment_config"], len(payload["labels"]), int(payload["experiment_config"]["num_frames"]))
net.load_state_dict(payload["state_dict"])
net.eval()
print({
    "model": payload["model_name"],
    "fold": "heldout_s1 (S1 unseen)",
    "seed": payload["seed"],
    "trainable_params": sum(p.numel() for p in net.parameters() if p.requires_grad),
    "classes": len(payload["labels"]),
})

pred_csv = Path("artifacts/runs/primary/heldout_s1/landmark_bilstm/seed_42/test_predictions.csv")
saved = pd.read_csv(pred_csv, keep_default_na=False)
sample = saved.sample(8, random_state=0)
records = []
with torch.no_grad():
    for row in sample.itertuples():
        array = np.load(Path("data/landmark_cache_primary") / f"{row.sample_id}.npy")
        logits = net(torch.from_numpy(array).unsqueeze(0))
        index = int(logits.argmax(1))
        records.append({
            "sample_id": row.sample_id,
            "true": row.true_label,
            "saved_pred": row.predicted_label,
            "replayed_pred": payload["labels"][index],
            "match_saved": payload["labels"][index] == row.predicted_label,
        })
replay = pd.DataFrame(records)
display(replay)
assert replay.match_saved.all()
print("Replayed logits match the committed test_predictions.csv rows.")


## 4. Confusion snapshot

Row-normalized mean confusion of the landmark BiLSTM over two folds and three seeds.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

matrix = pd.read_csv(
    "artifacts/results/confusion_landmark_bilstm_mean_normalized.csv",
    index_col=0,
    keep_default_na=False,
)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(matrix.to_numpy(dtype=float), cmap="mako", vmin=0, vmax=1)
ax.set_title("Landmark BiLSTM mean row-normalized confusion")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.show()

top = pd.read_csv("artifacts/results/top_confusions_landmark_bilstm.csv", keep_default_na=False).head(8)
display(top[["true_label", "predicted_label", "count_across_seeds_and_folds"]])


## 5. What this notebook does not claim

- **Not 93.5%.** That figure was a discarded placeholder and is not in these artifacts.
- **Not state of the art** on other NSL image or word datasets. Those papers used different files and splits.
- **Not a translator.** Unseen-signer accuracy is 24.8% ± 4.0% on 49 classes. Vowels are much easier than consonants here.
- **Not a Colab webcam study.** For a camera or a local `.MOV` file:

```bash
export PYTHONPATH=src
python scripts/webcam_demo.py \
  --checkpoint artifacts/runs/primary/heldout_s1/landmark_bilstm/seed_42/best_checkpoint.pt \
  --camera 0
```

On the Apple M4 used for the paper, the same script processed held-out NSL23 vowel clips at 33–42 FPS.

Full training (RGB + landmarks) and the LaTeX source are in the GitHub repository. RGB frame caches and the Zenodo zips are not in git; see `README.md` and `PROTOCOL.md`.
